In [4]:
import pycolmap
from pathlib import Path

# Set paths relative to where the script is executed
project_path = Path(".")
image_path = project_path / "images"
database_path = project_path / "database.db"
sparse_output_path = project_path / "sparse"

sparse_output_path.mkdir(exist_ok=True)

print("1. Extracting features...")
pycolmap.extract_features(database_path, image_path)

print("2. Matching features...")
pycolmap.match_exhaustive(database_path)

print("3. Running sparse reconstruction...")
maps = pycolmap.incremental_mapping(database_path, image_path, sparse_output_path)

print(f"Pipeline complete! Reconstructed {len(maps)} map(s).")

1. Extracting features...
2. Matching features...


: 

In [2]:
import cv2
import pycolmap
import os
from pathlib import Path

# --- Configuration ---
video_file = "my_video.mp4"         # Your video file name
extract_fps = 3                     # How many frames to extract per second of video
project_path = Path(".")
image_path = project_path / "images"
database_path = project_path / "database.db"
sparse_output_path = project_path / "sparse/ab"

# Create necessary directories
image_path.mkdir(exist_ok=True)
sparse_output_path.mkdir(exist_ok=True)

# --- Phase 1: Extract Frames from Video ---
print(f"Opening video: {video_file}...")
cap = cv2.VideoCapture(video_file)

if not cap.isOpened():
    print("Error: Could not open video file.")
    exit()

video_fps = cap.get(cv2.CAP_PROP_FPS)
frame_interval = int(video_fps / extract_fps)
frame_count = 0
saved_count = 0

print(f"Extracting {extract_fps} frames per second...")
while True:
    ret, frame = cap.read()
    if not ret:
        break # End of video
    
    # Only save frames based on our desired interval
    if frame_count % frame_interval == 0:
        # Format filename with leading zeros (e.g., frame_0001.jpg)
        frame_name = f"frame_{saved_count:04d}.jpg"
        cv2.imwrite(str(image_path / frame_name), frame)
        saved_count += 1
        
    frame_count += 1

cap.release()
print(f"Extracted {saved_count} images to the /images folder.")

# --- Phase 2: COLMAP Pipeline ---
print("\n1. Extracting features...")
pycolmap.extract_features(database_path, image_path)

print("\n2. Matching features (Sequential)...")
# CRITICAL DIFFERENCE: We use match_sequential instead of match_exhaustive.
# Because video frames are in order, COLMAP only needs to compare a frame 
# to the few frames immediately before and after it, saving massive amounts of time.
pycolmap.match_sequential(database_path)

print("\n3. Running sparse reconstruction...")
maps = pycolmap.incremental_mapping(database_path, image_path, sparse_output_path)

print(f"\nPipeline complete! Reconstructed {len(maps)} map(s).")

Opening video: my_video.mp4...
Extracting 3 frames per second...
Extracted 48 images to the /images folder.

1. Extracting features...

2. Matching features (Sequential)...

3. Running sparse reconstruction...

Pipeline complete! Reconstructed 1 map(s).


In [5]:

import open3d as o3d
import numpy as np
import pycolmap

# 1. Load the COLMAP reconstruction
model_path = "sparse/ab/0" 
reconstruction = pycolmap.Reconstruction(model_path)

print(f"Loaded model with {reconstruction.num_points3D()} points. Extracting data...")

# 2. Extract XYZ coordinates and RGB colors
points = []
colors = []

# Loop through the dictionary of points
for point_id, point3D in reconstruction.points3D.items():
    points.append(point3D.xyz)
    colors.append(point3D.color)

# Convert lists to NumPy arrays
xyz_coordinates = np.array(points)

# Open3D expects color values to be between 0.0 and 1.0, 
# but COLMAP stores them as 0 to 255, so we divide by 255.
rgb_colors = np.array(colors) / 255.0 

# 3. Build the Open3D PointCloud object
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(xyz_coordinates)
pcd.colors = o3d.utility.Vector3dVector(rgb_colors)

# Optional: Make the points a little easier to see by estimating normals
pcd.estimate_normals()

# 4. Launch the visualizer!
print("Launching Open3D visualizer... (Close the popup window to stop the script)")
o3d.visualization.draw_geometries([pcd], window_name="COLMAP Sparse Point Cloud")

Loaded model with 5800 points. Extracting data...
Launching Open3D visualizer... (Close the popup window to stop the script)
